In [ ]:
Content
1. Message Queue
    - Amazon SQS
    - Amazon SNS
2. kafka
3. Performance Definations
4. Microservices


# 1. Message Queue

In [ ]:
It enables asynchronous communication between services by storing messages in a queue until they are processed by consumers.
Message Queue = Async communication + Decoupling + Reliability + Load handling
              
Producer → Queue → Consumer
Produces, adds the message. Queue stores the message temporarily. Consumer process them async.

# Examples
Kafka → streaming, high throughput
SQS → simple managed queue (AWS)                                                                                        



In [ ]:
# Key Concepts
1. Visibility Timeout (SQS)
    - Message is hidden while processing
    - If consumer crashes → message becomes visible again
    - Workflow:
        1. Message in queue (visible)
        2. Consumer A reads message
        3. Message becomes invisible (say 30 sec)
        4. Consumer A processes it
        5a. Success → DELETE → done ✅
        5b. Failure → NOT deleted → becomes visible again ❗

2. Ordering
    - Normal queue → no strict order
    - FIFO queue → order guaranteed (slower)
3. Failure Handling
    1. Retry
        - Fail → Retry → Retry → Success
    2. Dead Letter Queue
        - Queue → Fail → Fail → Fail → DLQ
        - This is another Queue
        - maxReceiveCount = 5, After 5 failed attempts → message moves to DLQ
        - for debugging, DLQ → Inspect → Fix issue → Reprocess (optional)

    3. Idempotency
        - Process same message twice → same result
    4. Logging

# Amazon SQS
1. Retry: Retry is automatic — no need to push back to queue manually
    - If message is not deleted → it becomes visible again
2. Long Polling 
    - Reduce API calls and cost, and Improves Latency.
    - Additional Attribute: WaitTimeSeconds
        Without long polling:
            Request → No messages → Immediate empty response ❌
        
        With long polling (20 sec):
            Request → Wait up to 20 sec → Return when message arrives ✅

# SQS Configuration
Config                  | Meaning
------------------------|----------------------------------------
Queue Type              | Standard or FIFO
Visibility Timeout      | Time message stays invisible after read
Message Retention       | How long message stays in queue (1 min–14 days)
Delivery Delay          | Delay before message becomes visible
Max Message Size        | Up to 256 KB
Receive Wait Time       | Long polling (reduce empty calls)
DLQ                     | Dead letter queue config (maxReceiveCount = 5)

In [ ]:
# AWS Queue Messages:
1. Amazon SQS: Simple Queue Service
    - SQS = Queue = Work gets DONE
    - Similar to RabitMQ
    - Pull based queue, 
        - Producer sends message -> Queue
        - Consumer pulls message from Queue

2. Amazon SNS (Simple Notification Service)
    - It works on publisher-subscriber (pub/sub) model.
    - SNS = Notification = Message gets BROADCASTED
    - Similar to Kafka
    - Feature: Fan-out (1 → many)
    - Publisher sends message → Topic
	- SNS pushes message to all subscribers

👉 SNS + SQS together
	•	SNS receives event
	•	Fans out to multiple SQS queues
	•	Each service processes independently

Order Created → SNS Topic
   → SQS (Email Service)
   → SQS (Billing Service)
   → SQS (Analytics Service)



In [ ]:
# C# normal implementation
using System.Collections.Concurrent;
var queue = new ConcurrentQueue<string>();

// Producer
// Task.Run(() => )run code on a separate thread (thread pool) so that it executes asynchronously and concurrently with other work.
Task.Run(() =>
{
    for (int i = 0; i < 5; i++)
    {
        queue.Enqueue($"Message {i}");
        Console.WriteLine($"Produced: Message {i}");
        Thread.Sleep(500);
    }
});

// Consumer
Task.Run(() =>
{
    while (true)
    {
        if (queue.TryDequeue(out var message))
        {
            Console.WriteLine($"Consumed: {message}");
        }
    }
});



# Amazon SQS (Simple)
using Amazon.SQS;
using Amazon.SQS.Model;

var client = new AmazonSQSClient();

await client.SendMessageAsync(new SendMessageRequest
{
    QueueUrl = "your-queue-url",
    MessageBody = "order-created"
});

var response = await client.ReceiveMessageAsync(new ReceiveMessageRequest
{
    QueueUrl = "your-queue-url",
    MaxNumberOfMessages = 10,
    WaitTimeSeconds = 20
});

# Amazon SQS (Simple)

var queueUrl = "YOUR_QUEUE_URL";

var sendRequest = new SendMessageRequest
{
    QueueUrl = queueUrl,
    MessageBody = "Order Created: 123",
    
    // Optional: add metadata
    MessageAttributes = new Dictionary<string, MessageAttributeValue>
    {
        {
            "EventType", new MessageAttributeValue
            {
                DataType = "String",
                StringValue = "OrderCreated"
            }
        }
    }
};

var sendResponse = await client.SendMessageAsync(sendRequest);

Console.WriteLine($"Message Sent. ID: {sendResponse.MessageId}");


# 2. kafka

In [ ]:
# Event Sourcing
    - Instead of storing the current state, you store every change (event) that happened.
    - Event sourcing stores every state change as an immutable event in an append-only log, allowing the system to reconstruct state, maintain full history, and replay capabilities.
    - It creats EventLog (AppendOnly Log)
    - Problem without Event Sourcing
        - Let's say user uploads a video, and video requires multiple steps of processing (like converting to differnet format, different resolutions).
        - If we maintain a single field in DB to update the state, like (processing, completed). It is very likely, the updates might get lost as the DB might be down for that moment.
    - Example:
            {event: "VideoUpload", data:{ path: " }, timestamp }
            { event: "VideoProccessingInit", data:{ path: "}}
            {event: "VideoProccessingSucess", data:{ path: "

In [ ]:
Kafka → event streaming platform (log-based, replayable, high throughput)
SQS   → managed message queue (simple, reliable, task processing)

# Storing kafka messages
    - Kafka stores messages in a log (append only file)
    - Example:
        Offset → Message
        0      → OrderCreated
        1      → PaymentDone
        2      → OrderShipped
    - Replayable: Replay = re-reading old messages again whenever you want
    - Messages are NOT deleted after consumption
    - Consumers track their own offset
    - Different consumers may have different offset:
        - Consumer A reads from offset 0 → processes all messages
        - New Consumer B starts → can also read from offset 0 again ✅



# When Kafka should be used:
    Replay, High throughput, and multiple consumers
    1. You need event streaming
    2. You want replay capability
        - Kafka → YES (big advantage)
    3. High throughput is required
    4. Multiple consumers need same data
    5. Event-driven architecture (EDA)

# Comparison
1. SQS/ RabitMQ
    - Messages are deleted once consumed. Queue acts as a buffer.
    - Push-based. Broker pushes messages to consumers. Messages consumed once*

2. Kafka
    - Messages are retained for a configurable period regardless of consumption. Acts as a durable log.
    - Pull-based. Consumers poll for messages. Multiple consumers can independently read the same message by maintaining their own offset        
    - usually used where volume is high, and multiple consumers.

Feature              | Kafka                          | SQS
---------------------|--------------------------------|-------------------------
Type                 | Event streaming platform       | Message queue
Message Storage      | Persistent (log-based)         | Temporary (until deleted)
Replay               | Yes                            | No
Ordering             | Per partition                  | FIFO queues (optional)
Throughput           | Very high                      | Moderate
Scaling              | Manual (cluster)               | Auto (managed)
Ownership            | Self-managed / Confluent       | AWS managed
Use Case             | Event streaming, analytics     | Background jobs, async tasks
                                       
# Kafka Key terms
    - Offset-based consumption
    - Append-only log
    - Retention policy
    - Consumer groups
    - Event streaming

In Kafka, message deletion is done by N number of days, OR like when size reaches 1 GB.                    

# 3. Performance Definations

In [ ]:
1. High Throughput System
    - Throughput is amount of work a system can handle per unit time.
    - requests/ messages per seconds.
    - How to build such systems:
        1. Parallel Processing: Multiple requests in parallel using load balancers
        2. Async Processing: Message queues
        3. Data Handeling:
            - Batch DB calls
            - Caching (redis)
        4. DB
            - Partitioning

2. Latency:
    - Time taken for one request
            

# 4. Microservices

In [ ]:
Microservices, collection of small independent services, each responsible for specific business usecase.
 - Independent services that communicate over a network. They provide scalability, flexibility, and fault isolation, 
 - but introduce complexity in terms of distributed systems, data consistency, and debugging and Network latency.
 - They are best suited for large, scalable systems with multiple teams.
    
    
Client
  ↓
API Gateway
  ↓
---------------------------------
| Order Service                |
| Payment Service              |
| User Service                 |
| Notification Service         |
---------------------------------
  ↓
Each service has its own DB


# Key Characteristics
- Independent deployment and scaling
- Own database (no shared DB ideally)
- Communicate via API / messaging
- Loosely coupled
- Highly scalable
- Can use different tech stack
- Better fault toleration